In [1]:
# Cell 1 — confirm environment
import torch
import transformers, datasets, peft
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("peft:", peft.__version__)

torch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5060 Ti
transformers: 5.16.1
datasets: 5.0.1
peft: 0.20.0


In [2]:
# Cell 2 — load LEDGAR and inspect
from datasets import load_dataset

# LEDGAR: contract clause classification, 100 clause types.
# Part of the LexGLUE benchmark.
dataset = load_dataset("coastalcph/lex_glue", "ledgar")

print("Splits:", {k: len(v) for k, v in dataset.items()})
print("\nColumns:", dataset["train"].column_names)

# Look at one real example
example = dataset["train"][0]
print("\n--- Example clause ---")
print("Text:", example["text"][:400], "...")
print("Label (numeric):", example["label"])

# The 100 label names
label_names = dataset["train"].features["label"].names
print("\nNumber of labels:", len(label_names))
print("First 15 label names:", label_names[:15])

README.md:   0%|          | 0.00/34.1k [00:00<?, ?B/s]

d:\Projects\Legal Clause Classifier\legal-clause-classifier\venv\lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\AryanPC\.cache\huggingface\hub\datasets--coastalcph--lex_glue. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


ledgar/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.9MB            

ledgar/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

ledgar/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.31MB            

...

ledgar/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/60000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Splits: {'train': 60000, 'test': 10000, 'validation': 10000}

Columns: ['text', 'label']

--- Example clause ---
Text: Except as otherwise set forth in this Debenture, the Company, for itself and its legal representatives, successors and assigns, expressly waives presentment, protest, demand, notice of dishonor, notice of nonpayment, notice of maturity, notice of protest, presentment for the purpose of accelerating maturity, and diligence in collection. ...
Label (numeric): 97

Number of labels: 100
First 15 label names: ['Adjustments', 'Agreements', 'Amendments', 'Anti-Corruption Laws', 'Applicable Laws', 'Approvals', 'Arbitration', 'Assignments', 'Assigns', 'Authority', 'Authorizations', 'Base Salary', 'Benefits', 'Binding Effects', 'Books']


In [3]:
# Cell 3 — label distribution (decides our metric)
from collections import Counter

label_names = dataset["train"].features["label"].names
counts = Counter(dataset["train"]["label"])

# Most and least common clause types
sorted_counts = counts.most_common()
print("Most common clause types:")
for label_id, n in sorted_counts[:5]:
    print(f"  {label_names[label_id]:25} {n:>6}  ({100*n/60000:.1f}%)")

print("\nLeast common clause types:")
for label_id, n in sorted_counts[-5:]:
    print(f"  {label_names[label_id]:25} {n:>6}  ({100*n/60000:.1f}%)")

# Imbalance summary
most = sorted_counts[0][1]
least = sorted_counts[-1][1]
print(f"\nMost common: {most} | Least common: {least} | Imbalance ratio: {most/least:.1f}x")
print(f"Avg per class: {60000/100:.0f}")

Most common clause types:
  Governing Laws              3167  (5.3%)
  Notices                     2493  (4.2%)
  Counterparts                2427  (4.0%)
  Entire Agreements           2340  (3.9%)
  Severability                1808  (3.0%)

Least common clause types:
  Sanctions                    118  (0.2%)
  Anti-Corruption Laws         106  (0.2%)
  Qualifications                47  (0.1%)
  Assigns                       31  (0.1%)
  Books                         23  (0.0%)

Most common: 3167 | Least common: 23 | Imbalance ratio: 137.7x
Avg per class: 600


In [7]:
# Cell 4 — tokenize the dataset
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=384)

# Tokenize all splits
tokenized = dataset.map(tokenize, batched=True)
print("Tokenized. Columns now:", tokenized["train"].column_names)

# Check token-length distribution to confirm 384 is enough
import numpy as np
lengths = [len(tokenizer(t, truncation=False)["input_ids"]) for t in dataset["train"]["text"][:2000]]
print(f"Token lengths (sample of 2000): mean {np.mean(lengths):.0f}, "
      f"95th pct {np.percentile(lengths,95):.0f}, max {np.max(lengths)}")
print(f"Clauses over 384 tokens: {100*np.mean(np.array(lengths)>384):.1f}%")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (693 > 512). Running this sequence through the model will result in indexing errors


Tokenized. Columns now: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask']
Token lengths (sample of 2000): mean 145, 95th pct 385, max 1169
Clauses over 384 tokens: 5.1%


In [8]:
# Cell 5 — load DistilBERT for 100-class classification, wrap with LoRA
from transformers import AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType

label_names = dataset["train"].features["label"].names
num_labels = len(label_names)  # 100

id2label = {i: name for i, name in enumerate(label_names)}
label2id = {name: i for i, name in enumerate(label_names)}

base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q_lin", "v_lin"],
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 962,404 || all params: 67,992,776 || trainable%: 1.4155


In [10]:
# Cell 6 — train
import numpy as np
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding
from sklearn.metrics import f1_score, accuracy_score

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),      # headline (imbalance-aware)
        "weighted_f1": f1_score(labels, preds, average="weighted"),
    }

training_args = TrainingArguments(
    output_dir="models/distilbert-lora-ledgar",
    learning_rate=2e-4,               # LoRA tolerates higher LR than full fine-tuning
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=4,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    fp16=True,                        # mixed precision — faster on your GPU
    logging_steps=100,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,0.770491,0.663988,0.811800,0.705078,0.797232
2,0.582893,0.587102,0.834400,0.738682,0.823024
3,0.549630,0.557416,0.839100,0.752890,0.830221
4,0.489430,0.544234,0.843000,0.759602,0.834256


TrainOutput(global_step=7500, training_loss=0.7143015640258789, metrics={'train_runtime': 1041.5942, 'train_samples_per_second': 230.416, 'train_steps_per_second': 7.201, 'total_flos': 2.370403734896947e+16, 'train_loss': 0.7143015640258789, 'epoch': 4.0})

In [11]:
# Cell 7 — final evaluation on the held-out test set
test_results = trainer.evaluate(tokenized["test"])
print("=== Test set results (the honest, final numbers) ===")
for k, v in test_results.items():
    if any(m in k for m in ["accuracy", "f1", "loss"]):
        print(f"  {k}: {v:.4f}")

# Save these — they're your resume/README numbers
finetuned_metrics = {
    "accuracy": test_results["eval_accuracy"],
    "macro_f1": test_results["eval_macro_f1"],
    "weighted_f1": test_results["eval_weighted_f1"],
}
print("\nSaved finetuned_metrics:", finetuned_metrics)

Training Loss,Validation Loss,Epoch,Accuracy,Macro F1,Weighted F1
0.489430,0.554886,4,0.842900,0.749810,0.834416


=== Test set results (the honest, final numbers) ===
  eval_loss: 0.5549
  eval_accuracy: 0.8429
  eval_macro_f1: 0.7498
  eval_weighted_f1: 0.8344

Saved finetuned_metrics: {'accuracy': 0.8429, 'macro_f1': 0.7498097063021782, 'weighted_f1': 0.8344161223303549}


In [12]:
# Cell 8 — zero-shot Claude Haiku baseline setup
import os, time, json
from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()
client = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

HAIKU = "claude-haiku-4-5-20251001"
HAIKU_PRICING = (1.00, 5.00)  # $/M tokens (input, output)

# The 100 valid labels, numbered, for the prompt
labels_block = "\n".join(f"{i}: {name}" for i, name in enumerate(label_names))

SYSTEM = "You are a legal contract clause classifier. Classify each clause into exactly one of the given categories."

def classify_clause(text):
    """Zero-shot classify one clause. Returns (predicted_label_id or -1, cost, latency_ms)."""
    user = (
        f"Classify this contract clause into exactly ONE category by number.\n\n"
        f"CATEGORIES:\n{labels_block}\n\n"
        f"CLAUSE:\n{text[:1500]}\n\n"
        f"Respond with ONLY the category number (0-99). No other text."
    )
    start = time.time()
    msg = client.messages.create(
        model=HAIKU, max_tokens=10, system=SYSTEM,
        messages=[{"role": "user", "content": user}],
    )
    latency = (time.time() - start) * 1000
    cost = (msg.usage.input_tokens/1e6)*HAIKU_PRICING[0] + (msg.usage.output_tokens/1e6)*HAIKU_PRICING[1]

    # Parse the number; -1 if it returned something invalid
    raw = msg.content[0].text.strip()
    try:
        pred = int("".join(c for c in raw if c.isdigit())[:3] or "-1")
        if not (0 <= pred < 100):
            pred = -1
    except (ValueError, IndexError):
        pred = -1
    return pred, cost, latency

# Quick smoke test on 1 example
sample = dataset["test"][0]
pred, cost, lat = classify_clause(sample["text"])
print(f"True: {label_names[sample['label']]} ({sample['label']})")
print(f"Haiku predicted: {label_names[pred] if pred>=0 else 'INVALID'} ({pred})")
print(f"Cost: ${cost:.5f} | Latency: {lat:.0f}ms")

True: Employment (35)
Haiku predicted: Employment (35)
Cost: $0.00075 | Latency: 967ms


In [13]:
# Cell 9 — benchmark Haiku on a random 500-clause test sample
import random
random.seed(42)  # reproducible sample

N = 500
indices = random.sample(range(len(dataset["test"])), N)

haiku_preds, true_labels = [], []
total_cost, total_latency = 0.0, 0.0
invalid_count = 0

print(f"Classifying {N} clauses with zero-shot Haiku...\n")
for i, idx in enumerate(indices):
    ex = dataset["test"][idx]
    pred, cost, lat = classify_clause(ex["text"])
    haiku_preds.append(pred)
    true_labels.append(ex["label"])
    total_cost += cost
    total_latency += lat
    if pred == -1:
        invalid_count += 1
    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{N} done | running cost ${total_cost:.3f} | invalid so far: {invalid_count}")

# Save for the comparison
import numpy as np
haiku_preds = np.array(haiku_preds)
true_labels = np.array(true_labels)
print(f"\nDone. Total cost: ${total_cost:.3f} | avg latency: {total_latency/N:.0f}ms/clause")
print(f"Invalid labels returned: {invalid_count}/{N} ({100*invalid_count/N:.1f}%)")

Classifying 500 clauses with zero-shot Haiku...

  50/500 done | running cost $0.040 | invalid so far: 0
  100/500 done | running cost $0.080 | invalid so far: 0
  150/500 done | running cost $0.120 | invalid so far: 0
  200/500 done | running cost $0.159 | invalid so far: 0
  250/500 done | running cost $0.200 | invalid so far: 0
  300/500 done | running cost $0.240 | invalid so far: 0
  350/500 done | running cost $0.279 | invalid so far: 0
  400/500 done | running cost $0.320 | invalid so far: 0
  450/500 done | running cost $0.361 | invalid so far: 0
  500/500 done | running cost $0.400 | invalid so far: 0

Done. Total cost: $0.400 | avg latency: 652ms/clause
Invalid labels returned: 0/500 (0.0%)


In [15]:
# Cell 10 — score both models on the SAME 500 clauses
import torch
from sklearn.metrics import accuracy_score, f1_score

# --- Haiku metrics on the sample ---
haiku_acc = accuracy_score(true_labels, haiku_preds)
haiku_macro = f1_score(true_labels, haiku_preds, average="macro", labels=list(range(100)), zero_division=0)
haiku_weighted = f1_score(true_labels, haiku_preds, average="weighted", zero_division=0)

# --- Fine-tuned model on the SAME 500 clauses ---
model.eval()
device = "cuda"
ft_preds = []
with torch.no_grad():
    for idx in indices:
        text = dataset["test"][idx]["text"]
        inputs = tokenizer(text, truncation=True, max_length=384, return_tensors="pt").to(device)
        logits = model(**inputs).logits
        ft_preds.append(int(torch.argmax(logits, dim=-1).item()))
ft_preds = np.array(ft_preds)

ft_acc = accuracy_score(true_labels, ft_preds)
ft_macro = f1_score(true_labels, ft_preds, average="macro", labels=list(range(100)), zero_division=0)
ft_weighted = f1_score(true_labels, ft_preds, average="weighted", zero_division=0)

# --- The comparison table ---
print("="*60)
print(f"{'Metric':<20}{'Fine-tuned':<15}{'Zero-shot Haiku':<15}")
print("="*60)
print(f"{'Accuracy':<20}{ft_acc:<15.4f}{haiku_acc:<15.4f}")
print(f"{'Macro-F1':<20}{ft_macro:<15.4f}{haiku_macro:<15.4f}")
print(f"{'Weighted-F1':<20}{ft_weighted:<15.4f}{haiku_weighted:<15.4f}")
print(f"{'Latency/clause':<20}{'~2ms':<15}{'652ms':<15}")
print(f"{'Cost/clause':<20}{'~$0 (local)':<15}{'$0.0008':<15}")
print("="*60)

results_summary = {
    "sample_size": N,
    "finetuned": {"accuracy": ft_acc, "macro_f1": ft_macro, "weighted_f1": ft_weighted},
    "haiku": {"accuracy": haiku_acc, "macro_f1": haiku_macro, "weighted_f1": haiku_weighted,
              "cost_per_clause": total_cost/N, "latency_ms": total_latency/N, "invalid_rate": invalid_count/N},
}
print("\nSaved results_summary")

Metric              Fine-tuned     Zero-shot Haiku
Accuracy            0.8420         0.6880         
Macro-F1            0.7009         0.5073         
Weighted-F1         0.8282         0.6615         
Latency/clause      ~2ms           652ms          
Cost/clause         ~$0 (local)    $0.0008        

Saved results_summary


In [16]:
# Cell 11 — inference demo: paste a clause, get its predicted type
import torch

def classify(text, top_k=3):
    """Classify a clause; show the top-k predicted clause types with confidence."""
    model.eval()
    inputs = tokenizer(text, truncation=True, max_length=384, return_tensors="pt").to("cuda")
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=-1)[0]
    top = torch.topk(probs, top_k)
    print(f"Clause: {text[:150]}{'...' if len(text)>150 else ''}\n")
    print("Top predictions:")
    for score, idx in zip(top.values, top.indices):
        print(f"  {label_names[idx.item()]:25} {100*score.item():.1f}%")

# Try a few examples
print("="*55)
classify("This Agreement shall be governed by and construed in accordance with the laws of the State of Delaware.")
print("\n" + "="*55)
classify("Either party may terminate this Agreement upon thirty (30) days written notice to the other party.")
print("\n" + "="*55)
classify("The Company shall indemnify and hold harmless the Employee from any claims arising out of the performance of their duties.")

Clause: This Agreement shall be governed by and construed in accordance with the laws of the State of Delaware.

Top predictions:
  Governing Laws            92.9%
  Applicable Laws           6.5%
  Construction              0.2%

Clause: Either party may terminate this Agreement upon thirty (30) days written notice to the other party.

Top predictions:
  Terminations              99.0%
  Terms                     0.4%
  Notices                   0.3%

Clause: The Company shall indemnify and hold harmless the Employee from any claims arising out of the performance of their duties.

Top predictions:
  Indemnifications          85.7%
  Indemnity                 13.8%
  General                   0.2%
